# Mean Reversion Strategy - 3D Visualization

## Overview
This notebook demonstrates a mean reversion trading strategy using Apple (AAPL) stock data with interactive 3D visualization.

## Strategy
- **Moving Average**: 20-day simple moving average
- **Z-Score**: Measures how far the price deviates from the mean
- **Signal**: Buy when z-score < -2 (oversold), Sell when z-score > 2 (overbought)

In [22]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go

## Data Processing & Calculation

### Steps:
1. Download AAPL stock data (2022-2024)
2. Calculate 20-day moving average
3. Compute rolling standard deviation
4. Calculate z-score: `(price - MA20) / std`
5. Remove NaN values from initial calculations

In [23]:
# Download stock data from Yahoo Finance
data = yf.download("AAPL", start="2022-01-01", end="2024-01-01", auto_adjust=True)

# Calculate mean reversion indicators
data["MA20"] = data[("Close", "AAPL")].rolling(window=20).mean()
data["std"] = data[("Close", "AAPL")].rolling(window=20).std()
data["zscore"] = (data[("Close", "AAPL")] - data["MA20"]) / data["std"]

# Remove NaN values
data = data.dropna()

# Create time index for plotting
data["time_index"] = range(len(data))

[*********************100%***********************]  1 of 1 completed


## Interactive 3D Visualization

### Axes:
- **X-axis**: Time (trading days)
- **Y-axis**: Stock price
- **Z-axis**: Z-score (deviation from mean)

### Interpretation:
- **Blue line**: Stock price movement over time with z-score
- **Red dashed line**: Mean reversion level (z-score = 0)
- **When z-score > 2**: Stock is overbought (potential sell signal)
- **When z-score < -2**: Stock is oversold (potential buy signal)

In [24]:
# Create interactive 3D visualization
fig = go.Figure(data=[
    go.Scatter3d(
        x=data["time_index"],
        y=data[("Close", "AAPL")],
        z=data["zscore"],
        mode='lines',
        line=dict(width=4, color='blue'),
        name='AAPL Mean Reversion Path'
    )
])

# Update layout for better visualization
fig.update_layout(
    title="Interactive 3D Mean Reversion Visualization - AAPL (2022-2024)",
    scene=dict(
        xaxis_title='Time (Trading Days)',
        yaxis_title='Stock Price ($)',
        zaxis_title='Z-Score'
    ),
    width=1000,
    height=700,
    font=dict(size=12)
)

# Add reference plane at z=0 for mean reversion level
fig.add_trace(go.Scatter3d(
    x=data["time_index"],
    y=data[("Close", "AAPL")],
    z=[0] * len(data),
    mode='lines',
    line=dict(width=2, color='red', dash='dash'),
    name='Mean Level (Z=0)',
    showlegend=True
))

fig.show()

## Summary Statistics

### Key Metrics:
- Total trading days analyzed
- Price range during the period
- Z-score range (volatility around mean)

### Performance Insights:
- Larger z-score values indicate higher volatility
- Frequent mean reversion suggests good trading opportunities

In [25]:
# Display summary statistics
print(f"Data points: {len(data)}")
print(f"Date range: {data.index[0].strftime('%Y-%m-%d')} to {data.index[-1].strftime('%Y-%m-%d')}")
print(f"Price range: ${data[('Close', 'AAPL')].min():.2f} - ${data[('Close', 'AAPL')].max():.2f}")
print(f"Z-score range: {data['zscore'].min():.2f} to {data['zscore'].max():.2f}")

Data points: 482
Date range: 2022-01-31 to 2023-12-29
Price range: $123.05 - $196.07
Z-score range: -3.04 to 2.50
